#### Notebook to process corpus specifically for Domingo's project

## Map Domingo's authors and institutions to OpenAlex correspondences

## Compute the weights for citations

## Prepare plot andtable for reputational citations

## Expore co-authorship and cross-citation for Domingo's authors

In [19]:
%run common_setup.ipynb

#### MatchFactory matches two strings using exact, acronym, short and fuzzy

In [20]:
import glob
import re
from unidecode import unidecode
from polyfuzz import PolyFuzz
from polyfuzz.models import RapidFuzz, TFIDF, EditDistance, Embeddings
from flair.embeddings import TransformerWordEmbeddings, WordEmbeddings
from jellyfish import jaro_winkler_similarity

class MatchFactory:

    def __init__(self):
        self.factory = {
            'exact': self._exact_matcher,
            'acronym': self._acronym_matcher,
            'short': self._short_matcher,
            'fuzzy': self._fuzzy_matcher
            }
        return

    def match_factory(self, from_list, to_list):
        self.from_list = from_list
        self.to_list = to_list
        match = {}
        for match_type in self.factory.keys():
            print(f'{len(from_list) = } {len(to_list) = }')
            match  |= self.factory[match_type](from_list, to_list)
            # print(match)
            from_list = [from_name for from_name in from_list if from_name not in match.keys()]
            to_list = [to_name for to_name in to_list if to_name not in match.values()]
            if not from_list or not to_list:
                return match
        return match

    def _exact_matcher(self, from_list, to_list):
        exact_match = []
        exact_match.extend(item for item in from_list if item in to_list)
        # print(f'{len(exact_match) = }/{len(from_list) = }')
        return dict(zip(exact_match, exact_match))

    def _acronym_matcher(self, from_list, to_list):
        acronym_match = {}
        for from_name in from_list:
            if m := re.search(r'([A-Z]+?){3,}', from_name):
                match = m[0]
                if match not in ['USA', 'UK']:
                    # print(f'{from_name = } {match = }')
                    for to_name in to_list:
                        if match == to_name: # or match in to_name:
                            # print(f'== {from_name = } {match = } {to_name = }')
                            acronym_match[from_name] = to_name
        return acronym_match

    def _short_matcher(self, from_list, to_list):
        to_dict = dict(zip(to_list, self.tidy_list(to_list)))
        short_match = {}
        for from_name, from_short in zip(from_list, self.tidy_list(from_list)):
            # print(f'{from_name = } {from_short = }')
            for to_name, to_short in to_dict.items():
                # print(f'{to_name = } {to_short = }')
                if from_short in to_short:
                    short_match[from_name] = to_name
                    break
        return short_match

    def _fuzzy_matcher(self, from_list, to_list):
        tfidf = TFIDF(n_gram_range=(3,3), min_similarity=0.5, model_id="TF-IDF")
        rapid_fuzz = RapidFuzz(n_jobs=1, score_cutoff=0.8, model_id='RapidFuzz')
        matchers = [tfidf, rapid_fuzz]
        model = PolyFuzz(matchers)
        model.match(from_list, to_list) #self.tidy_list(from_list), to_list)
        match = pd.concat(list(model.get_matches().values()), axis=0).sort_values('Similarity', ascending=False).drop_duplicates(subset=['From'])
        # print(match.tail(32))
        return {f: t for f, t, s in zip(match.From, match.To, match.Similarity) if s > 0.8 and t != None}

    def tidy_list(self, in_list):
        out_list = []
        for item in in_list:
            parts = [part.strip().lower() for part in unidecode(item).split(' ')]
            out_list.append(' '.join([p for p in parts if p not in ['the', 'of', '&', 'and', 'de', '-', 'universite', 'universidade', 'universitat', 'university']])) #'university'
        return out_list

#### Match to Domingo's 600 institutions using MatchMaker

start with
- len(from_list) = 600 len(to_list) = 79472

match exact names
- len(from_list) = 175 len(to_list) = 79033

match acronyms
len(from_list) = 149 len(to_list) = 78980

match shortened names (the, of, university etc)
- len(from_list) = 64 len(to_list) = 78894

match finally with fuzzy

In [21]:
class MatchDomingoInstitutions(SetUp, MatchFactory):

    def __init__(self):
        super().__init__()
        return

    def extract_institutions(self):
        # Extract journals or institutions from Domingo and load into duckdb in memory
        # The only information is the name. There are around 600 of each.
        # For journals, hopefully we can use the WOS JCR to map names to ISSN
        # For institutions we use the MatchMaker()
        df = pd.read_excel('../DATA/eco_bus_inst_journal_scores.xlsx', sheet_name='institutions')
        df = df.drop(columns=['Unnamed: 0'])
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE memory.institutions AS (SELECT * FROM df)")
        self.db.sql("SELECT count(*) FROM memory.institutions").show()
        return
    
    def institution_matcher(self):

        df = self.db.sql("SELECT * FROM memory.institutions").df() #.iloc[:64]
        
        oa = self._extract_institutions()
        oa_id_dict = dict(zip(oa.institution_name, oa.institution_id))
        oa_count_dict = dict(zip(oa.institution_name, oa.works_count))
        print(f'{oa.shape = }\n{oa.head()}')
        print(oa.loc[oa.institution_name.str.contains('INSEAD')].head(99))

        from_list = df.institution.to_list()
        to_list = oa.institution_name.to_list()
        matched = MatchFactory().match_factory(from_list, to_list)
        
        df.insert(0, 'institution_id', [oa_id_dict.get(matched.get(from_name)) for from_name in df.institution])
        df.insert(1, 'institution_name', [matched.get(from_name) for from_name in df.institution])
        df.insert(2, 'works_count', [oa_count_dict.get(matched.get(from_name)) for from_name in df.institution])
        print(df[df.institution_id.isnull()].head())
        self.db.sql("CREATE OR REPLACE TABLE project.institutions_matched AS SELECT * FROM df")
        self.db.sql("SELECT count(*) FROM project.institutions_matched").show()
        self.db.sql("SELECT * FROM project.institutions_matched").show()
        return

    def _extract_institutions(self):
        sql = """
            -- ETL institutions FROM raw.authorships JOIN works.institutions
            -- =============================================================
            SELECT DISTINCT institution_id,
                    institution_name,
                    a.country_code,
                    list_append(i.display_name_alternatives, i.display_name) AS display_name_alternatives,
                    count(DISTINCT work_id) AS works_count
            FROM project.authorships a
            INNER JOIN institutions.institutions i
            ON i.id = a.institution_id
            GROUP BY ALL
        """
        return self.db.sql(sql).df()

#### This cell matches Domingo's C and T (and CT) lists to authors in the OpenAlex corpus
#### It also creates the X sample from OpenAlex  

- Extract Domingo's list and ensure that the names are normalised
- Match Domingo' Names to OpenAlex display_names
- Allocate the T, C, TC and X samples

In [22]:
class MatchDomingoAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_and_load_sample(self):
        sample = pd.read_excel('../DATA/researchers_results_total_average_influence_MASTER.xlsx').iloc[:, :10]
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        # print(sample[sample.duplicated(keep=False)].head(32))
        self.db.sql("CREATE OR REPLACE TABLE project.domingo_sample_original AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM project.domingo_sample_original").df()
        print(f'{self.sample.shape = }\n{self.sample.head()}')
        return

    def sample_author_matcher(self):
        sql = """
            -- ETL TO MATCH Domingo's Research_Profile TO OpenAlex display_name_alternatives
            -- =============================================================================
            CREATE OR REPLACE TABLE project.domingo_sample_matched AS
            WITH
                author_name_matches_CTE AS
                (SELECT author_id, d.*
                    FROM project.domingo_sample_original d
                    LEFT JOIN project.authors_alt
                    USING (fullname)
                    -- WHERE author_id IS NULL
                ),
                include_author_data_CTE AS
                (SELECT DISTINCT * 
                    FROM author_name_matches_CTE
                    LEFT JOIN project.authors
                    USING (author_id)
                )

            SELECT *
            FROM
                (SELECT *,
                        first_value(works_count_endogenous) OVER (PARTITION BY ACR ORDER BY works_count_endogenous DESC) AS top_works_count
                FROM include_author_data_CTE
                )
            WHERE (works_count_endogenous = top_works_count)
            ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.domingo_sample_matched").show()
        return
    
    def sample_extender(self):  # sourcery skip: identity-comprehension

        sql = """
            -- ETL TO CREATE A sample TABLE from DOMINGO + OpenAlex high producers
            -- ===================================================================
            CREATE OR REPLACE TABLE project.sample AS
            SELECT *
                FROM
                (SELECT count(author_name) OVER (PARTITION BY author_name) AS dupes,
                        *
                    FROM 
                    (
                        (SELECT *
                        FROM project.domingo_sample_matched)
                        
                        UNION BY NAME
                        
                        (SELECT a.author_id, a.author_name, a.works_count, a.cited_by_count, a.h_index, a.h_index_2yr, 'X' AS "Group", 'OA' AS "Class"
                            FROM project.authors a
                            LEFT JOIN project.domingo_sample_matched
                            USING (author_id)
                            -- WHERE "Group" IS NULL
                            ORDER BY a.cited_by_count_endogenous DESC
                            LIMIT 130
                        )

                        UNION BY NAME

                        (SELECT a.author_id, a.author_name, a.works_count, a.cited_by_count, a.h_index, a.h_index_2yr, 'Y' AS "Group", 'OA' AS "Class"
                            FROM project.authors a
                            LEFT JOIN project.domingo_sample_matched
                            USING (author_id)
                            -- WHERE "Group" IS NULL
                            ORDER BY a.works_count_endogenous DESC
                            LIMIT 130
                        )

                        UNION BY NAME

                        (SELECT author_id, author_name, works_count, cited_by_count, h_index, h_index_2yr, "Group", "Class", 
                            FROM (SELECT a.author_id, a.author_name, a.works_count, a.cited_by_count, a.h_index, a.h_index_2yr, 'Z' AS "Group", 'OA' AS "Class", 
                                        a.cited_by_count_endogenous/a.works_count_endogenous AS ratio 
                                    FROM project.citation_summary a
                                    LEFT JOIN project.domingo_sample_matched
                                    USING (author_id)
                                    -- WHERE "Group" IS NULL
                                )
                            ORDER BY ratio DESC
                            LIMIT 130
                        )
                    )
                )
                -- WHERE dupes = 1 OR Research_Profile NOT NULL
                ORDER BY "Group", h_index_2yr DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.sample").show()
        return

## Cell to assemble the citation weights drawing on source/journal and institution scores from Domingo.

#### Get works-specific citation weights

#### Combine weights to get author_level weighted citations.

In [23]:
class BuildReputationCitations(SetUp):

    def __init__(self):
        super().__init__()
        return

    def source_institution_weights_for_works(self):
        sql = """
            -- ETL TO BUILD source AND institution WEIGHTS for each WORK pending weighting the CITATIONS
            -- =========================================================================================
            CREATE OR REPLACE TABLE project.citation_weight AS
            WITH
                sources_CTE AS
                (SELECT *
                    FROM project.sources_matched
                ),
                works_sources_CTE AS
                (SELECT id AS work_id,
                        source_id,
                        journal_score
                    FROM project.raw
                    LEFT JOIN sources_CTE
                    USING (source_id)
                ),
                institutions_CTE AS
                (SELECT institution_id,
                        iscore
                    FROM project.institutions_matched
                ),
                works_institutions_CTE AS
                (SELECT work_id,
                        -- count(institution_id),
                        -- count(),
                        -- sum(institution_score),
                        sum(institution_score)/count() AS institution_score_weighted
                    FROM 
                    (SELECT DISTINCT work_id,
                            institution_id,
                            CASE WHEN institution_id IS NULL THEN 1
                                WHEN iscore IS NULL THEN 1
                                ELSE iscore END AS institution_score
                        FROM project.authorships
                        LEFT JOIN project.institutions_matched
                        USING (institution_id)
                        )
                    GROUP BY work_id
                    )

            SELECT DISTINCT work_id,
                    journal_score,
                    institution_score_weighted,
                    (journal_score+institution_score_weighted)/2 AS combined_weight
            FROM works_sources_CTE
            LEFT JOIN works_institutions_CTE
            USING (work_id)
            ORDER BY combined_weight ASC
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.citation_weight").show()
        return

    def build_author_citation_report(self):
        sql = """  
            -- ETL TO COMPUTE AN AUTHORS cited_by_count AND cited_by_count_weighted BY GROUPING CITING WORKS
            -- =============================================================================================
            CREATE OR REPLACE TABLE project.author_citation_report AS
                WITH
                citations_CTE AS
                    (SELECT cited_id AS work_id,
                            mean(delta_t) AS delta_t_mean,
                            count(citer_id) AS cited_by_count,
                            round(sum(combined_weight), 4) AS cited_by_count_weighted,
                            cited_by_count_weighted/cited_by_count AS citation_ratio
                    FROM project.citer_cited
                    LEFT JOIN project.citation_weight
                    ON work_id = citer_id
                    WHERE work_id IS NOT NULL
                    GROUP BY ALL
                    )
            
            SELECT *
                FROM project.sample s
                LEFT JOIN (SELECT DISTINCT
                                author_id,
                                author_name,
                                sum(cited_by_count) AS citations,
                                sum(cited_by_count_weighted) AS citations_weighted,
                                sum(cited_by_count_weighted)/sum(cited_by_count) AS citation_ratio
                            FROM project.authorships
                            LEFT JOIN citations_CTE
                            USING (work_id)
                            WHERE author_id NOT NULL
                            GROUP BY ALL
                            ) c
                USING (author_id)
                ORDER BY citation_ratio DESC    
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.author_citation_report").show()
        return

In [24]:
class SamplePlotter(SetUp):

    def __init__(self, flag=None):
        super().__init__()
        return

    def loader(self):
        data = self.db.sql("SELECT * FROM project.author_citation_report").df()
        print(f'{data.shape = }\n{data.head()}')
        return data

    def plot_one(self, data):
        print(data.groupby('Group')['Group'].count())
        print(f'{data.shape = }\n{data.head()}')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
        sns.scatterplot(data, x='citations', y='citations_weighted',ax=ax1, hue='Group', hue_order=['T', 'C', 'X', 'Y', 'Z'], size='Group', sizes=[20, 20, 5, 5, 5], size_order=['T', 'C', 'X', 'Y', 'Z'])
        ax1.set_xscale('log')
        ax1.set_xlim(10.0, 250000)
        ax1.set_yscale('log')
        ax1.set_ylim(10.0, 250000)
        ax1.set_title("Author's citations and weighted citations")
        sns.scatterplot(data, x='citations', y='citation_ratio', ax=ax2, hue='Group', hue_order=['T', 'C', 'X', 'Y', 'Z'] , size='Group',sizes=[20, 20, 5, 5, 5] , size_order=['T', 'C', 'X', 'Y', 'Z'])
        ax2.set_xscale('log')
        ax2.set_xlim(10.0, 250000)
        ax2.set_title("Author's citations and weight ratio")
        plt.tight_layout()
        plt.savefig("../PLOTS/citations_and_weighted_citations.png", dpi=300)
        plt.show()

    def plot_facets(self, data):
        df = data[['Group', 'citations', 'citation_ratio']]
        df = df.dropna()
        df['Group'] = df['Group'].map({'C': 'Control', 'T': 'inCites HCR', 'Z': 'OAx HCA', 'X': 'OAx top work counts', 'Y': 'OAx top cite counts'})
        flag_values = df['Group'].unique()[:5]
        flag_values = ['Control', 'inCites HCR', 'OAx HCA', 'OAx top work counts', 'OAx top cite counts']
        print(f'{flag_values = }')
        first_flag = flag_values[0]

        # Prepare data for overlay: first category and each of the next three
        overlay_data = df[df['Group'] == first_flag]
        fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
        axes = axes.flatten()
        for i, other_flag in enumerate(flag_values[1:]):
            facet_data = df[df["Group"] == other_flag]
            plt.figure()
            # Plot the other category
            sns.scatterplot(data=facet_data, x='citations', y='citation_ratio', ax=axes[i], label=other_flag, color='C0')
            # Overlay the first category
            sns.scatterplot(data=overlay_data, x='citations', y='citation_ratio', ax=axes[i], label=first_flag, color='C1')
            axes[i].set_title(f"{other_flag} (overlay: {first_flag})")
            axes[i].legend()
            axes[i].set_xscale('log')
            axes[i].set_xlim(10.0, 250000)
            axes[i].set_xscale('log')
            axes[i].set_ylim(0.0, 3.0)
        plt.tight_layout()
        plt.show()

    

#### This class builds CSV files for Domingo made up from the OA data

In [25]:
class TablesForDomingo(SetUp):

    # builds a dictionarty of tables to save as CSVs for Domingo

    def __init__(self):
        super().__init__()
        self.assemble_tables = {}
        return
    
    def match_table(self):
        sql = """  
            -- MATCH Domingo's EconBus list to OpenAlex
            -- ========================================
            SELECT *
            FROM project.domingo_sample_matched
        """
        sample = self.db.sql(sql).df()
        sample = sample.sort_values(['Group', 'Research_Profile'], ascending=[False, True]).reset_index(drop=True)
        print(f'{sample.shape = }\n{sample.head()}')
        self.assemble_tables |= {'Domingo_matched': sample}
        return
    
    def works_table(self):
        sql = """
            -- ETL works for Domingo
            -- =====================
            SELECT id as work_id,
                    doi, 
                    title, 
                    publication_year,
                    fwci,
                    cited_by_count,
                    referenced_works_count,
                    "primary_location.source".display_name AS source_name,
                    "primary_location.source".issn_l AS issn,
                    "primary_location.source".host_organization_name AS publisher,
                    "biblio.volume" AS volume,
                FROM project.raw
            """
        works = self.db.sql(sql).df()
        works = works.sort_values(['publication_year', 'fwci'], ascending=[True, False]).reset_index(drop=True)
        print(f'{works.shape = }\n{works.head()}')
        self.assemble_tables |= {'works': works}
        return
    
    def authorships_table(self):
        sql = """
            -- ETL FOR Domingo authors and institutions
            -- ========================================
            SELECT work_id,
                    author_name,
                    author_id,
                    institution.display_name AS institution_name,
                    institution.id AS institution_id,
                    institution.country_code AS country_code
            FROM
                (SELECT work_id,
                    authorship.author.display_name as author_name,
                    authorship.author.id AS author_id,
                    unnest(authorship.institutions) AS institution,
                FROM 
                    (SELECT id AS work_id,
                        unnest(authorships) AS authorship
                    FROM project.raw
                    )
                )
                """
        authorships = self.db.sql(sql).df()
        print(f"{authorships.shape = }\n{authorships.head()}")
        self.assemble_tables |= {'authorships': authorships}
        return
    
    def references_table(self):
        sql = """
            -- ETL FOR Domingo reference_list
            -- ==============================
            SELECT id AS work_id,
                    referenced_works
                FROM project.raw
            """
        references = self.db.sql(sql).df()
        print(f'{references.shape = }\n{references.head()}')
        self.assemble_tables |= {'references': references}
        return
    
    def citations_table(self):
        sql = """
            -- ETL FOR Domingo citation_list
            -- =============================
            SELECT cited_id,
                    list(citer_id) AS citing_works_list
            FROM 
                (SELECT id AS citer_id,
                    unnest(referenced_works) AS cited_id
                FROM project.raw
                )
            GROUP BY ALL
            """
        citations = self.db.sql(sql).df()
        print(f'{citations.shape = }\n{citations.head()}')
        self.assemble_tables |= {'citations': citations}
        return
    
    def topics_table(self):
        sql = """ 
            -- ETL FOR Domingo topics
            -- ======================
            SELECT id AS work_id,
                    "primary_topic.display_name" AS topic_name,
                    "primary_topic.subfield".display_name AS subfield_name,
                    "primary_topic.field".display_name AS field_name,
                    "primary_topic.domain".display_name AS domain_name,
                    "primary_topic.score" AS topic_score
            FROM project.raw
            """
        topics = self.db.sql(sql).df()
        print(f'{topics.shape = }\n{topics.head()}')
        self.assemble_tables |= {'topics': topics}
        return


    def load_tables(self):
        for k, v in self.assemble_tables.items():
            with open(f'../DATA/tables_for_Domingo_{k}.csv', 'w') as writer:
                print(f'{k = } {v.shape = }\n{v.head()}')
                v.to_csv(writer, index=False)


In [26]:
def main():

    mdi = MatchDomingoInstitutions()
    mdi.extract_institutions()
    mdi.institution_matcher()
    mdi.db.close()

    mdas = MatchDomingoAuthors()
    mdas.extract_and_load_sample()
    mdas.sample_author_matcher()
    mdas.sample_extender()
    mdas.db.close()

    brc = BuildReputationCitations()
    brc.source_institution_weights_for_works()
    brc.build_author_citation_report()
    brc.db.close()

    sp = SamplePlotter()
    data = sp.loader()
    # sp.plot_one(data)
    sp.plot_facets(data)
    
    # tfd = TablesForDomingo()
    # tfd.match_table()
    # tfd.works_table()
    # tfd.authorships_table()
    # tfd.references_table()
    # tfd.citations_table()
    # tfd.topics_table()
    # tfd.load_tables()
    # tfd.db.close()

    return

In [27]:
if __name__ == "__main__":
    main()
    print("DONE")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ author_citation_re…  │ [dupes, author_id,…  │ [BIGINT, VARCHAR, VARCHAR, VARC…  │ false     │
│ project      │ main    │ autho

BinderException: Binder Error: Table "a" does not have a column named "works_count"

Candidate bindings: : "works_count_total", "works_count_endogenous"

LINE 39: ...                      FROM (SELECT a.author_id, a.author_name, a.works_count, a.cited_by_count, a.h_index, a.h_index_2yr...
                                                                           ^